# Phoenix nano on Colab — verification only

This notebook runs the **nano** ranking (and optional retrieval) path from the public [xAI x-algorithm](https://github.com/xai-org/x-algorithm) Phoenix export. It is a **toy verification** of install + synthetic data + a 6-step train + a ranking smoke serve. It does **not** prove production quality, production performance, or that you have production checkpoints.

**What this is:** the shipped nano configs (`home_direct_packed_nano` ranking, `xrecsys_two_tower_nano` retrieval) on ~12k synthetic rows, following [`phoenix/QUICKSTART.md`](https://github.com/xai-org/x-algorithm/blob/main/phoenix/QUICKSTART.md).

**What this is not:** production Phoenix (2560-d / 8-layer ranking), real user data, or xAI serving infrastructure. No production checkpoints ship in the repo.

### Runtime (required)

1. **Runtime → Change runtime type → Hardware accelerator → GPU**
2. Pick **A100** or **H100** (Colab Pro / Pro+). H200 / GB200 / GB300 are also accepted if they appear.
3. Free-tier **Tesla T4** (and L4) are the **wrong architecture**. Phoenix `gpu_arch()` only knows A100 / H100 / H200 / GB200 / GB300, and nano ranking's default attention is Hopper-tuned Pallas (`pallas_ranker_varlen_attn`). The next cell aborts on T4 / L4 / CPU.

Official Phoenix requirement: **Linux + NVIDIA GPU + CUDA 12**. Colab GPU runtimes are that OS family.

Do not run this on a Mac host. Do not `pip install jax` into Colab's system Python. Phoenix pins `jax[cuda12]==0.8.1` inside its uv environment; **every** training and serving command is `uv run` from `phoenix/`.

Keep this tab focused: the first `uv sync --extra engine` can take **10–30+ minutes**, and Colab idle-disconnects around ~90 minutes on free.


In [ ]:
# Fail fast: default path requires A100 / H100 (H200 / GB200 / GB300 also ok).
import shutil
import subprocess

def nvidia_smi_L() -> str:
    if shutil.which("nvidia-smi") is None:
        print("nvidia-smi not found (CPU runtime, or NVIDIA tools missing).")
        return ""
    try:
        return subprocess.check_output(["nvidia-smi", "-L"], text=True, timeout=30)
    except (subprocess.CalledProcessError, FileNotFoundError, OSError) as exc:
        print(f"nvidia-smi -L failed: {exc}")
        return ""

listing = nvidia_smi_L()
print(listing if listing else "(no GPU listing)")

ALLOWED = ("A100", "H100", "H200", "GB200", "GB300")
upper = listing.upper()
if not any(name in upper for name in ALLOWED):
    raise SystemExit(
        "GPU gate failed. Phoenix default kernels are Hopper-tuned; "
        "gpu_arch() only recognizes A100 / H100 / H200 / GB200 / GB300. "
        "Free-tier Tesla T4 is the wrong architecture; L4 is also unsupported "
        "on the default path. Do not continue the default cells. "
        "Runtime → Change runtime type → GPU → A100 or H100. "
        "A later markdown cell notes what a T4/L4 experiment would require "
        "(attn_impl=jax_attn, smaller batch; may still OOM)."
    )

print("GPU gate passed. Detected:")
print(listing.strip())


## System packages, Rust, protoc, uv

Colab already has Python, but Phoenix's `engine` extra builds a Rust gRPC engine (maturin). That needs a C/C++ toolchain, cmake, RDMA verbs headers, libclang, libnuma, Rust, and `protoc` ≥ 3.15 (the protos use proto3 `optional`; Ubuntu's `protobuf-compiler` is often 3.12).

Install uv with the official installer. Do **not** `pip install jax` into the notebook kernel.


In [ ]:
%%bash
set -euo pipefail
sudo apt-get update
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y \
  build-essential ca-certificates cmake curl pkg-config unzip git \
  libibverbs-dev libnl-3-dev libnl-route-3-dev libclang-dev libnuma-dev


In [ ]:
%%bash
set -euo pipefail
if ! command -v rustc >/dev/null 2>&1; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
fi
# shellcheck source=/dev/null
source "$HOME/.cargo/env"
rustc --version
cargo --version


In [ ]:
%%bash
set -euo pipefail
arch="$(uname -m)"
case "$arch" in
  x86_64) proto_arch="linux-x86_64" ;;
  aarch64) proto_arch="linux-aarch_64" ;;
  *) echo "unsupported uname -m: $arch"; exit 1 ;;
esac
curl -fsSL -o /tmp/protoc.zip \
  "https://github.com/protocolbuffers/protobuf/releases/download/v28.3/protoc-28.3-${proto_arch}.zip"
sudo unzip -o /tmp/protoc.zip -d /usr/local 'bin/*' 'include/*'
hash -r
protoc --version


In [ ]:
%%bash
set -euo pipefail
if [ ! -x "$HOME/.local/bin/uv" ]; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
fi
export PATH="$HOME/.local/bin:$PATH"
uv --version


In [ ]:
# Persist tool PATH / PYTHONPATH for later cells. Re-run this if the kernel restarts
# after the install cells. Do not `import jax` in this kernel.
import os
import subprocess
import time
from pathlib import Path

REPO = Path("/content/x-algorithm")
PHOENIX = REPO / "phoenix"


def ensure_tool_path() -> None:
    extras = [
        str(Path.home() / ".cargo" / "bin"),
        str(Path.home() / ".local" / "bin"),
        "/usr/local/bin",
    ]
    parts = os.environ.get("PATH", "").split(":")
    for extra in extras:
        if extra not in parts:
            parts.insert(0, extra)
    os.environ["PATH"] = ":".join(parts)


ensure_tool_path()


def run(cmd: str, cwd=None, tail: int = 30) -> None:
    # Heartbeat while the command runs, then print the last `tail` log lines.
    ensure_tool_path()
    cwd = str(cwd or os.getcwd())
    print(f"+ {cmd}\n  cwd={cwd}", flush=True)
    log_path = Path("/tmp/phoenix_colab_cmd.log")
    log_f = log_path.open("w", encoding="utf-8")
    try:
        env = os.environ.copy()
        env.setdefault("PYTHONUNBUFFERED", "1")
        proc = subprocess.Popen(
            ["bash", "-c", f"set -euo pipefail; {cmd}"],
            cwd=cwd,
            stdout=log_f,
            stderr=subprocess.STDOUT,
            env=env,
            text=True,
        )
        t0 = time.time()
        while proc.poll() is None:
            time.sleep(20)
            try:
                nlines = log_path.read_text(encoding="utf-8", errors="replace").count("\n")
            except OSError:
                nlines = 0
            print(
                f"  still running ({int(time.time() - t0)}s, {nlines} log lines)...",
                flush=True,
            )
    finally:
        log_f.close()
    text = log_path.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()
    show = 80 if proc.returncode else tail
    if len(lines) > show:
        print(f"... ({len(lines) - show} earlier lines omitted)")
    print("\n".join(lines[-show:]))
    if proc.returncode != 0:
        raise RuntimeError(
            f"exit {proc.returncode}: {cmd}\n"
            "Full log: /tmp/phoenix_colab_cmd.log"
        )


for tool in ("uv", "rustc", "cargo", "protoc"):
    print(
        subprocess.check_output(
            ["bash", "-c", f"command -v {tool}; {tool} --version"],
            text=True,
            env=os.environ.copy(),
        )
    )


In [ ]:
%%bash
set -euo pipefail
if [ -d /content/x-algorithm/phoenix ]; then
  echo "Reusing existing /content/x-algorithm"
else
  git clone --depth 1 https://github.com/xai-org/x-algorithm.git /content/x-algorithm
fi
test -f /content/x-algorithm/phoenix/QUICKSTART.md
ls /content/x-algorithm/phoenix | head


In [ ]:
import os
from pathlib import Path

REPO = Path("/content/x-algorithm")
PHOENIX = REPO / "phoenix"
if not (PHOENIX / "pyproject.toml").is_file():
    raise SystemExit(f"missing {PHOENIX / 'pyproject.toml'}; clone cell failed")

%cd /content/x-algorithm/phoenix
os.chdir(PHOENIX)
os.environ["PYTHONPATH"] = str(PHOENIX)
print("cwd:", os.getcwd())
print("PYTHONPATH:", os.environ["PYTHONPATH"])


## `uv sync --extra engine`

This builds the Rust gRPC engine (maturin) and installs Phoenix-pinned JAX **0.8.1** inside the uv virtualenv — not Colab's system JAX. First run: **10–30+ minutes**. Keep the tab focused.

Do not `import jax` in this notebook kernel; Colab's JAX is a different install and the wrong pin.


In [ ]:
# Long: engine extra compiles the Rust gRPC server. Last 30 log lines are shown.
# Requires the PATH helper cell (defines run, PHOENIX). Keep the tab focused.
os.chdir(PHOENIX)
os.environ["PYTHONPATH"] = str(PHOENIX)
run("uv sync --extra engine", cwd=PHOENIX, tail=30)
run(
    "uv run python -c 'import jax; print(\"jax\", jax.__version__); print(jax.devices())'",
    cwd=PHOENIX,
    tail=20,
)


## Smoke-test ranking (random weights)

No checkpoint or data needed. Boots the serving stack with **nano** random weights.

`bench.py --smoke` without `--config_name` defaults to production `home_direct_packed_aggregated_kafka` (100M embedding tables). `bench.py` also forces `ep=1`, so that config does not fit on a 40GB A100 and the child `launch_inference.py` dies before gRPC is ready. Use the nano config below. You can skip this cell and go to synthetic data + 6-step train if you prefer.


In [ ]:
os.chdir(PHOENIX)
run(
    "uv run python xrex/inference/oss_bench/bench.py --smoke --service_type ranking "
    "--config_name home_direct_packed_nano_offline_kafka_dump",
    cwd=PHOENIX,
    tail=40,
)


## Synthetic data

Same seed as QUICKSTART (`20260721`): world snapshots (SID + multimodal + post-creation), then a 12,288-row training dump. Sets `PHOENIX_INDEX_BASE` for later cells.


In [ ]:
os.chdir(PHOENIX)
run(
    "uv run python reference/world_snapshots.py --out ./synth_index --seed 20260721",
    cwd=PHOENIX,
    tail=30,
)
os.environ["PHOENIX_INDEX_BASE"] = str(PHOENIX / "synth_index")
print("PHOENIX_INDEX_BASE =", os.environ["PHOENIX_INDEX_BASE"])


In [ ]:
os.chdir(PHOENIX)
os.environ["PHOENIX_INDEX_BASE"] = str(PHOENIX / "synth_index")
run(
    "uv run python reference/dump_gen.py --out ./synth_dump --seed 20260721 "
    "--num-rows 12288 --partitions 4 --rows-per-file 1024 "
    "--sid ./synth_index/sid_snapshot/post_sid_v5_256x6.parquet --self-check",
    cwd=PHOENIX,
    tail=30,
)


## Train nano ranking (6 steps)

Quickstart scale, not the `train_synth.py` default of 500. Default config is `home_direct_packed_nano_offline_kafka_dump`. Six steps on synthetic data are a plumbing check, not model quality.

The first JAX compile of `jit_update` on an A100 is slow (several minutes, `slow_operation_alarm` is expected). After that, training died in **metrics logging**: `peak_tflops()` knows A100 as an architecture but had no TFLOPS entry, so MFU crashed after the compile. The next cell patches `xrex/utils/gpu.py` on this Colab clone (GitHub main is missing the A100 row), then trains. Re-run is much faster because XLA cache remains.


In [ ]:
os.chdir(PHOENIX)
os.environ["PYTHONPATH"] = str(PHOENIX)
os.environ["PHOENIX_INDEX_BASE"] = str(PHOENIX / "synth_index")

# Colab clones GitHub main, which lists A100 in gpu_arch() but omits it from
# the peak-TFLOPS table. handle_metrics() then asserts after the first step.
gpu_py = PHOENIX / "xrex" / "utils" / "gpu.py"
src = gpu_py.read_text(encoding="utf-8")
needle = """_GPU_ARCH_TO_PEAK_TFLOPS: dict[GpuArch, float] = {
    GpuArch.H100: 989.5,"""
patch = """_GPU_ARCH_TO_PEAK_TFLOPS: dict[GpuArch, float] = {
    GpuArch.A100: 312.0,
    GpuArch.H100: 989.5,"""
if "GpuArch.A100: 312.0" in src:
    print("peak_tflops already has A100")
elif needle not in src:
    raise RuntimeError(f"could not patch {gpu_py}: unexpected contents")
else:
    gpu_py.write_text(src.replace(needle, patch, 1), encoding="utf-8")
    print("patched", gpu_py, "with A100 312 TFLOPS")

run(
    'uv run python reference/train_synth.py '
    '--data ./synth_dump --steps 6 --out "$PWD/checkpoints" --metrics',
    cwd=PHOENIX,
    tail=40,
)


In [ ]:
import json

os.chdir(PHOENIX)
metrics_path = PHOENIX / "checkpoints" / "run" / "metrics.jsonl"
if not metrics_path.is_file():
    raise SystemExit(f"missing {metrics_path}; ranking train cell did not write metrics")

rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
print(f"{len(rows)} row(s) in {metrics_path}")
print(f"{'step':>8}  {'loss':>14}")
print("-" * 26)
for m in rows:
    step = m.get("step", "")
    loss = m.get("loss", "")
    try:
        print(f"{step:>8}  {float(loss):>14.6g}")
    except (TypeError, ValueError):
        print(f"{step:>8}  {loss!s:>14}")


## Optional: train nano retrieval (6 steps)

Skip if you only want ranking-only `bench.py`. Required for the retrieve→rank loop. Same dump; writes under `checkpoints/xrecsys_two_tower_nano_offline_kafka_dump/`. Retrieval checkpoints also embed the candidate index.

Set `TRAIN_RETRIEVAL = True` in the next cell.


In [ ]:
TRAIN_RETRIEVAL = False  # set True to train nano retrieval (another JAX compile + 6 steps)

os.chdir(PHOENIX)
os.environ["PYTHONPATH"] = str(PHOENIX)
os.environ["PHOENIX_INDEX_BASE"] = str(PHOENIX / "synth_index")

if not TRAIN_RETRIEVAL:
    print("Skipping retrieval training. Set TRAIN_RETRIEVAL = True and re-run this cell.")
else:
    run(
        'uv run python reference/train_synth.py '
        '--config xrecsys_two_tower_nano_offline_kafka_dump '
        '--data ./synth_dump --steps 6 --out "$PWD/checkpoints"',
        cwd=PHOENIX,
        tail=30,
    )


## Serve: ranking bench against the checkpoint (default)

QUICKSTART §4. `bench.py` restores the ranking checkpoint, starts gRPC, sends one synthetic request, then stops the server. Prefer this over retrieve→rank in Colab (background servers are fragile). Wait for port 9988 to be released before any later full-stack section.


In [ ]:
os.chdir(PHOENIX)
os.environ["PYTHONPATH"] = str(PHOENIX)
os.environ["PHOENIX_INDEX_BASE"] = str(PHOENIX / "synth_index")
run(
    '''RANK_CKPT=$(ls -d "$PWD"/checkpoints/home_direct_packed_nano_offline_kafka_dump/elapsed_samples_*/*/ | sort | tail -1)
echo "RANK_CKPT=$RANK_CKPT"
test -n "$RANK_CKPT"
uv run python xrex/inference/oss_bench/bench.py \\
  --checkpoint_path "$RANK_CKPT" \\
  --service_type ranking \\
  --config_name home_direct_packed_nano_offline_kafka_dump
''',
    cwd=PHOENIX,
    tail=30,
)


## Optional: retrieve → rank (QUICKSTART §5)

Colab background processes are fragile (output capture, idle disconnect, VRAM). The default path above is ranking-only bench.

If you have VRAM/time, trained retrieval, and want the full loop: SID on `:50061`, retrieval `:9990`, ranking `:9988`, then `retrieve_then_rank.py`. Launch flags match QUICKSTART §5, including `XLA_PYTHON_CLIENT_MEM_FRACTION=0.30` and `fake_mm_embeddings true`.

Set `RUN_RETRIEVE_THEN_RANK = True` in the next cell **after** retrieval training succeeded. Expect `retrieve_then_rank: 3 session(s) completed the full loop.`


In [ ]:
RUN_RETRIEVE_THEN_RANK = False  # requires TRAIN_RETRIEVAL; Colab background jobs are fragile

import os
import subprocess
import time
from glob import glob
from pathlib import Path

os.chdir(PHOENIX)
os.environ["PYTHONPATH"] = str(PHOENIX)
os.environ["PHOENIX_INDEX_BASE"] = str(PHOENIX / "synth_index")


def newest_ckpt(pattern: str) -> str:
    hits = sorted(glob(pattern))
    if not hits:
        raise SystemExit(f"no checkpoint matched {pattern}")
    return hits[-1]


def start_logged(cmd: str, log_file: Path, extra_env=None):
    ensure_tool_path()
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    if extra_env:
        env.update(extra_env)
    log_file.parent.mkdir(parents=True, exist_ok=True)
    log_f = log_file.open("w", encoding="utf-8")
    proc = subprocess.Popen(
        ["bash", "-c", f"set -euo pipefail; {cmd}"],
        cwd=str(PHOENIX),
        stdout=log_f,
        stderr=subprocess.STDOUT,
        env=env,
        start_new_session=True,
        text=True,
    )
    return proc, log_f


if not RUN_RETRIEVE_THEN_RANK:
    print("Skipping retrieve→rank. Set RUN_RETRIEVE_THEN_RANK = True after retrieval training.")
else:
    logs = Path("/tmp/phoenix_servers")
    logs.mkdir(parents=True, exist_ok=True)
    run("pkill -f 'sid_index_server.py|launch_inference.py' || true", cwd=PHOENIX, tail=5)
    time.sleep(2)

    rank_ckpt = newest_ckpt(
        str(PHOENIX / "checkpoints/home_direct_packed_nano_offline_kafka_dump/elapsed_samples_*/*/")
    )
    retr_ckpt = newest_ckpt(
        str(PHOENIX / "checkpoints/xrecsys_two_tower_nano_offline_kafka_dump/elapsed_samples_*/*/")
    )
    print("RANK_CKPT=", rank_ckpt)
    print("RETR_CKPT=", retr_ckpt)

    sid_cmd = (
        "uv run python reference/sid_index_server.py "
        "--parquet ./synth_index/sid_snapshot/post_sid_v5_256x6.parquet --port 50061"
    )
    # Flags match QUICKSTART.md section 5 (including fake_mm_embeddings true).
    retr_cmd = (
        "uv run python xrex/inference/launch_inference.py"
        " --driver local --service_type retrieval"
        " --config_name xrecsys_two_tower_nano_offline_kafka_dump"
        f" --checkpoint_path '{retr_ckpt}' --grpc_port 9990"
        " --sid_endpoint localhost:50061"
        " --num_devices_per_process 1 --bs_per_device 1"
        " --history_seq_len 128 --candidate_seq_len 8"
        " --max_inflight_requests 16 --allow_random_init false --fake_mm_embeddings true"
        " attn_impl=pallas_ranker_attn use_seqpack=False right_anchored_rope=True"
        " bs_per_device=1 parallel_config.num_devices_per_process=1 num_devices_per_process=1"
        " ep=1 dp=1 training_ep=1"
    )
    rank_cmd = (
        "uv run python xrex/inference/launch_inference.py"
        " --driver local --service_type ranking"
        " --config_name home_direct_packed_nano_offline_kafka_dump"
        f" --checkpoint_path '{rank_ckpt}' --grpc_port 9988 --metrics_port 9091"
        " --num_devices_per_process 1 --bs_per_device 1"
        " --history_seq_len 128 --candidate_seq_len 16"
        " --max_inflight_requests 16 --allow_random_init false --fake_mm_embeddings true"
        " attn_impl=pallas_ranker_attn_infer use_seqpack=False right_anchored_rope=True"
        " bs_per_device=1 parallel_config.num_devices_per_process=1 num_devices_per_process=1"
        " ep=1 dp=1 training_ep=1"
        " model_config.model_config.sequence_len=146"
    )

    xla_env = {"XLA_PYTHON_CLIENT_MEM_FRACTION": "0.30"}
    # Keep Popen + file handles alive so GC does not close server stdout.
    server_handles = [
        start_logged(sid_cmd, logs / "sid.log"),
        start_logged(retr_cmd, logs / "retrieval.log", extra_env=xla_env),
        start_logged(rank_cmd, logs / "ranking.log", extra_env=xla_env),
    ]
    print("server pids:", [p.pid for p, _fh in server_handles])

    deadline = time.time() + 600
    retr_txt = rank_txt = ""
    while time.time() < deadline:
        retr_txt = (logs / "retrieval.log").read_text(encoding="utf-8", errors="replace") if (logs / "retrieval.log").exists() else ""
        rank_txt = (logs / "ranking.log").read_text(encoding="utf-8", errors="replace") if (logs / "ranking.log").exists() else ""
        ready = {
            "retrieval": "Server ready to serve" in retr_txt,
            "ranking": "Server ready to serve" in rank_txt,
        }
        print("ready:", ready, flush=True)
        if all(ready.values()):
            break
        time.sleep(15)
    else:
        print("retrieval log tail:\n" + "\n".join(retr_txt.splitlines()[-30:]))
        print("ranking log tail:\n" + "\n".join(rank_txt.splitlines()[-30:]))
        raise SystemExit("servers did not become ready in 10 minutes")

    run(
        "uv run python reference/retrieve_then_rank.py "
        "--data ./synth_dump --sessions 3 --topk 16 "
        "--retrieval-port 9990 --ranking-port 9988",
        cwd=PHOENIX,
        tail=30,
    )


## Not the default path: Tesla T4 / L4

<!--
Phoenix gpu_arch() only recognizes A100 / H100 / H200 / GB200 / GB300.
Nano ranking defaults to attn_impl=pallas_ranker_varlen_attn (Hopper-tuned Pallas).

A T4/L4 experiment would need at least:
  - attn_impl=jax_attn instead of the Pallas ranker kernels
  - a smaller bs_per_device than nano's 64
  - and may still OOM on Colab free-tier VRAM

Do not make this the main flow. The GPU gate cell aborts on T4/L4 for a reason.
-->

Free-tier T4 (and L4) are the wrong architecture for the default nano kernels. The GPU gate is supposed to abort. If you fork this notebook anyway, you would need `attn_impl=jax_attn`, a smaller batch, and you may still OOM. That is unsupported here.


## What you just ran

| | This notebook (nano toy) | Production Phoenix |
|---|---|---|
| Ranking width / depth | 512-d, 4 layers | 2560-d, 8 layers |
| Retrieval width / depth | 512-d, 4 layers | 1024-d, 8 layers |
| Data | synthetic dump, 12,288 rows, seed 20260721 | production Kafka / real sessions |
| Steps | 6 (quickstart plumbing) | orders of magnitude more |
| Checkpoints | trained here under `phoenix/checkpoints/` | **not shipped** |
| Attention (ranking train) | `pallas_ranker_varlen_attn` (Hopper) | same family, cluster-scale |

You verified that the public export can install (`uv sync --extra engine`), generate a synthetic world, train nano ranking for 6 steps, print `checkpoints/run/metrics.jsonl`, and serve ranking via `bench.py`. That is not evidence of recommendation quality, production latency, or scale.


In [ ]:
# Optional Drive persistence (leave commented). Colab disks are ephemeral.
#
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copytree(
#     "/content/x-algorithm/phoenix/checkpoints",
#     "/content/drive/MyDrive/phoenix_nano_checkpoints",
#     dirs_exist_ok=True,
# )
